In [0]:
from pytickersymbols import PyTickerSymbols
import yfinance as yf
from pyspark.sql.types import StructType, StructField, DateType, DoubleType, LongType, StringType, BooleanType
import pandas as pd

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS yfinance_pipeline_prod;
USE CATALOG yfinance_pipeline_prod;
CREATE SCHEMA IF NOT EXISTS stocks_dataset;
USE SCHEMA stocks_dataset;

In [0]:
stock_data = PyTickerSymbols()

dowjones_stocks = stock_data.get_stocks_by_index('DOW JONES')
dowjones_symbols = [stock['symbol'] for stock in dowjones_stocks]
nasdaq100_stocks = stock_data.get_stocks_by_index('NASDAQ 100')
nasdaq100_symbols = [stock['symbol'] for stock in nasdaq100_stocks]
sp500_stocks = stock_data.get_stocks_by_index('S&P 500')
sp500_symbols = [stock['symbol'] for stock in sp500_stocks]
# sp600_stocks = stock_data.get_stocks_by_index('S&P 600')
# sp600_symbols = [stock['symbol'] for stock in sp600_stocks]


In [0]:
# Collect unique symbols from all major stock lists
collected_symbols = []

# for symbol_list in [nasdaq100_symbols, dowjones_symbols, sp500_symbols, sp600_symbols]:
for symbol_list in [nasdaq100_symbols, dowjones_symbols, sp500_symbols]:
    for symbol in symbol_list:
        if symbol not in collected_symbols and "." not in symbol:
            collected_symbols.append(symbol)

print(f"Total unique symbols: {len(collected_symbols)}")
print(collected_symbols[:20])  # Show first 20 for preview

In [0]:
def load_stocks_financial_statement(
    collected_symbols,
    balancesheet_table="yfinance_pipeline_prod.stocks_dataset.balance_sheet",
    income_table="yfinance_pipeline_prod.stocks_dataset.income_statement",
    cashflow_table="yfinance_pipeline_prod.stocks_dataset.cash_flow",
    batch_size=100
):
    for i in range(0, len(collected_symbols), batch_size):

        current_batch = collected_symbols[i:i + batch_size]
        print(f"Processing batch {i//batch_size + 1}: {current_batch[0]} to {current_batch[-1]}...")

        tickers = yf.Tickers(current_batch)

        bs_dfs = []
        is_dfs = []
        cf_dfs = []

        for symbol in current_batch:
            try:
                ticker_obj = tickers.tickers[symbol]

                statements = {
                    "balance_sheet": ticker_obj.get_balance_sheet(freq="quarterly"),
                    "income_statement": ticker_obj.get_income_stmt(freq="quarterly"),
                    "cash_flow": ticker_obj.get_cashflow(freq="quarterly")
                }

                for statement_type, df in statements.items():

                    if df is None or df.empty:
                        continue

                    df_reset = df.reset_index().rename(columns={"index": "metrics"})

                    df_long = df_reset.melt(
                        id_vars=["metrics"],
                        var_name="date",
                        value_name="value"
                    )

                    df_long["symbol"] = symbol
                    df_long["date"] = pd.to_datetime(df_long["date"]).dt.date
                    df_long["value"] = df_long["value"].fillna(0)

                    df_long = df_long[["symbol", "metrics", "date", "value"]]

                    if statement_type == "balance_sheet":
                        bs_dfs.append(df_long)

                    elif statement_type == "income_statement":
                        is_dfs.append(df_long)

                    elif statement_type == "cash_flow":
                        cf_dfs.append(df_long)

            except Exception as e:
                print(f"Error fetching {symbol}: {e}")

        # Write each table per batch

        if bs_dfs:
            bs_batch = spark.createDataFrame(pd.concat(bs_dfs, ignore_index=True))
            bs_batch.write.format("delta").mode("overwrite").saveAsTable(balancesheet_table)

        if is_dfs:
            is_batch = spark.createDataFrame(pd.concat(is_dfs, ignore_index=True))
            is_batch.write.format("delta").mode("overwrite").saveAsTable(income_table)

        if cf_dfs:
            cf_batch = spark.createDataFrame(pd.concat(cf_dfs, ignore_index=True))
            cf_batch.write.format("delta").mode("overwrite").saveAsTable(cashflow_table)

        print(f"Finished batch {i//batch_size + 1}")


In [0]:
if not spark.catalog.tableExists("yfinance_pipeline_prod.stocks_dataset.balance_sheet"):
    load_stocks_financial_statement(collected_symbols=collected_symbols)